In [1]:
import pandas as pd
import gradio as gr
import pathlib
import os
from ultralytics import YOLO
from PIL import Image
import xgboost as xgb

In [2]:
def crop_img(image: Image.Image):
    image_width, image_height = image.size

    tile_width = image_width
    tile_height = image_height
    if (image_width < image_height):
        tile_width = tile_width // 2
    else:
        tile_height = tile_height // 2

    tiles = []
    for x in range(0, image_width, tile_width):
        for y in range(0, image_height, tile_height):
            left = x
            upper = y
            right = x + tile_width
            lower = y + tile_height

            tile = image.crop((left, upper, right, lower))
            tiles.append(tile)
            # tile.save(f'tile_{x}_{y}.jpg')

    return tiles

In [3]:
cards_remaining = 200
run_count = 0
true_count = 0

def create_dataframe(player_cards, dealer_cards):
    if (11 in player_cards and sum(player_cards) > 21):
        for i in range(0, len(player_cards)):
            if player_cards[i] == 11 and sum(player_cards) > 21:
                player_cards[i] = 1
    player_value = sum(player_cards)
    
    vector_cartas = [0] * 8
    for i, carta in enumerate(player_cards[:8]):
        vector_cartas[i] = carta

    datos_jugada = {
        "cards_remaining": [cards_remaining],
        "dealer_up": [dealer_cards],
        "player_final_action_value": [player_value],
        "run_count": [run_count],
        "true_count": [true_count],
    }
    for i in range(8):
        datos_jugada[f"carta_{i+1}"] = [vector_cartas[i]]
    
    return pd.DataFrame(datos_jugada)

In [4]:
example_list = [["Examples/" + example] for example in os.listdir("Examples")]

card_recognition_model = YOLO("../CardRecognitionModel/runs/detect/cards_detector/weights/best.pt")
blackjack_model = xgb.XGBClassifier()
blackjack_model.load_model("../BlackjackModel/Models/modelo_blackjack.json")

card_value_dict = {"01": 11, "02": 2, "03": 3, "04": 4, "05": 5, "06": 6, "07": 7, "08": 8, "09": 9, "10": 10, "J": 10, "Q": 10, "K": 10, "background": 0}
action_dict = {0: "Double down", 1: "Hit", 2: "No insurance", 3: "Split", 4: "Surrender", 5: "Stand"}

def predict(
    img: Image.Image
):
    # Cortamos la imagen a la mitad, siempre se cortará por el lado más largo
    tiles = crop_img(img)

    # Pasamos ambas imágenes por el modelo de reconocimiento de cartas
    card_recognition_results = []
    for tile in tiles:
        res = card_recognition_model(tile)

        identified_img = res[0]
        boxes = identified_img.boxes
        names = identified_img.names
        
        predicted_cards = [names[int(c)] for c in boxes.cls.tolist()]
        predicted_cards = [card_value_dict[i] for i in predicted_cards]
        card_recognition_results.append((predicted_cards, identified_img))

    # Con los resultados del reconocimiento de cartas, averiguamos cual de las dos mitades tiene una sola carta, ya que será la del crupier
    if (len(card_recognition_results[0][0]) > len(card_recognition_results[1][0])):
        dealer = card_recognition_results[1][0][0]
        dealer_img = card_recognition_results[1][1]
        player = card_recognition_results[0][0]
        player_img = card_recognition_results[0][1]
    else:
        dealer = card_recognition_results[0][0][0]
        dealer_img = card_recognition_results[0][1]
        player = card_recognition_results[1][0]
        player_img = card_recognition_results[1][1]

    df_jugada = create_dataframe(player, dealer)

    pred = blackjack_model.predict(df_jugada)[0]
    pred = action_dict[pred]


    return pred, dealer_img.plot(), player_img.plot()

interface = gr.Interface(predict, gr.Image(type="pil"), [gr.Textbox(label="Acción recomendada"), gr.Image(label="Crupier"), gr.Image(label="Jugador")], examples=example_list)
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
